In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr
from datasets import load_dataset
import sacrebleu
import speech_recognition as sr
from typing import List, Tuple, Optional
import yaml
import warnings
warnings.filterwarnings('ignore')

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
print(f"GPU properties: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

GPU properties: 6.44 GB


In [10]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

# Use config values
print(config["messages"]["loading"])
print(config["messages"]["waiting"])

Loading model and tokenizer...
Please wait while the model is being loaded. This may take a few moments.


In [28]:
tokenizer = AutoTokenizer.from_pretrained(
    config["model"]["name"],
    src_lang=config["model"]["src_lang"],
    use_fast=config["model"]["use_fast_tokenizer"]
)

DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host='huggingface.co', port=443): Failed to resolve 'huggingface.co' ([Errno 11001] getaddrinfo failed)"))' thrown while requesting HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (2): huggingface.co:443
'HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host='huggingface.co', port=443): Failed to resolve 'huggingface.co' ([Errno 11001] getaddrinfo failed)"))' thrown while requesting HEAD https://huggingface.co/facebook/nllb-200-dis

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

DEBUG:filelock:Attempting to release lock 1547319622640 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\e316b82de11d0f951f370943b3c438311629547285129b0b81dadabd01bca665.lock
DEBUG:filelock:Lock 1547319622640 released on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\e316b82de11d0f951f370943b3c438311629547285129b0b81dadabd01bca665.lock
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): huggingface.co:443
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/added_tokens.json HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/special_tokens_map.json HTTP/1.1" 307 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/special_tokens_map.json HTTP/1.1" 200 0
DEBU

special_tokens_map.json: 0.00B [00:00, ?B/s]

DEBUG:filelock:Attempting to release lock 1547254482416 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\02fba84e401dd2017999a39cac2c1278591c3a03.lock
DEBUG:filelock:Lock 1547254482416 released on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\02fba84e401dd2017999a39cac2c1278591c3a03.lock


In [30]:
MODEL_NAME = config["model"]["name"]

In [31]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/config.json HTTP/1.1" 307 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json HTTP/1.1" 200 0
DEBUG:filelock:Attempting to acquire lock 1547135713152 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\91ba54326eccfdb17a06a7d110df3d71f39b84b8.lock
DEBUG:filelock:Lock 1547135713152 acquired on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\91ba54326eccfdb17a06a7d110df3d71f39b84b8.lock
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json HTTP/1.1" 200 846


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

DEBUG:filelock:Attempting to release lock 1547135713152 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\91ba54326eccfdb17a06a7d110df3d71f39b84b8.lock
DEBUG:filelock:Lock 1547135713152 released on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\91ba54326eccfdb17a06a7d110df3d71f39b84b8.lock
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/model.safetensors HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/model.safetensors.index.json HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/pytorch_model.bin HTTP/1.1" 302 0
DEBUG:filelock:Attempting to acquire lock 1547320097536 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\c266c2cfd19758b6d09c1fc31ecdf1e485509035f6b

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

DEBUG:filelock:Attempting to release lock 1547320097536 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\c266c2cfd19758b6d09c1fc31ecdf1e485509035f6b51dfe84f1ada83eefcc42.lock
DEBUG:filelock:Lock 1547320097536 released on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\c266c2cfd19758b6d09c1fc31ecdf1e485509035f6b51dfe84f1ada83eefcc42.lock
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): huggingface.co:443
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/model.safetensors HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /facebook/nllb-200-distilled-600M/resolve/main/generation_config.json HTTP/1.1" 307 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d1

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

DEBUG:filelock:Attempting to release lock 1547480656896 on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\4c6d168201607a22255bee55a6ed092b0e3ed7a1.lock
DEBUG:filelock:Lock 1547480656896 released on C:\Users\bengu\.cache\huggingface\hub\.locks\models--facebook--nllb-200-distilled-600M\4c6d168201607a22255bee55a6ed092b0e3ed7a1.lock


In [32]:
print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Model dtype: {model.dtype}")

Model loaded successfully!
Model parameters: 615.1M
Model dtype: torch.float16


In [33]:
# Language code mapping (FLORES-200 format)
LANGUAGE_CODES = {
    "English": "eng_Latn",
    "French": "fra_Latn",
    "Arabic": "arb_Arab",
    "Spanish": "spa_Latn",
    "German": "deu_Latn",
    "Chinese (Simplified)": "zho_Hans",
    "Chinese (Traditional)": "zho_Hant",
    "Japanese": "jpn_Jpan",
    "Korean": "kor_Hang",
    "Russian": "rus_Cyrl",
    "Portuguese": "por_Latn",
    "Italian": "ita_Latn",
    "Dutch": "nld_Latn",
    "Turkish": "tur_Latn",
    "Polish": "pol_Latn",
    "Hindi": "hin_Deva",
    "Bengali": "ben_Beng",
    "Urdu": "urd_Arab",
    "Vietnamese": "vie_Latn",
    "Thai": "tha_Thai",
    "Indonesian": "ind_Latn",
    "Malay": "zsm_Latn",
    "Swahili": "swh_Latn",
    "Greek": "ell_Grek",
    "Hebrew": "heb_Hebr",
    "Persian": "pes_Arab",
    "Ukrainian": "ukr_Cyrl",
    "Czech": "ces_Latn",
    "Swedish": "swe_Latn",
    "Danish": "dan_Latn",
    "Finnish": "fin_Latn",
    "Norwegian": "nob_Latn",
    "Hungarian": "hun_Latn",
    "Romanian": "ron_Latn",
    "Bulgarian": "bul_Cyrl",
    "Croatian": "hrv_Latn",
    "Serbian": "srp_Cyrl",
    "Slovak": "slk_Latn",
    "Lithuanian": "lit_Latn",
    "Latvian": "lvs_Latn",
    "Estonian": "est_Latn",
    "Slovenian": "slv_Latn",
    "Catalan": "cat_Latn",
    "Tagalog": "tgl_Latn",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr"
}

# Get list of supported languages
SUPPORTED_LANGUAGES = list(LANGUAGE_CODES.keys())

print(f"Configured {len(SUPPORTED_LANGUAGES)} languages")
print(f"\nSample languages: {', '.join(SUPPORTED_LANGUAGES[:10])}...")

Configured 50 languages

Sample languages: English, French, Arabic, Spanish, German, Chinese (Simplified), Chinese (Traditional), Japanese, Korean, Russian...
